## DamageDiffusion - Colab Training

---

### Configuration

In [ ]:
GITHUB_REPO = "https://github.com/youlkar/damage-diffusion.git"
PROJECT_DIR = "/content/drive/MyDrive/DamageDiffusion"
NUM_EPOCHS = 100

### Mount Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
os.makedirs(PROJECT_DIR, exist_ok=True)

print("Google Drive mounted")
print(f"Project directory: {PROJECT_DIR}")

### Verify GPU

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name}")
    print(f"VRAM: {gpu_memory:.1f} GB")
    
    # Auto-detect optimal batch size
    if 'T4' in gpu_name:
        BATCH_SIZE = 12
        print("Recommended batch size: 12 (T4)")
    elif 'V100' in gpu_name:
        BATCH_SIZE = 24
        print("Recommended batch size: 24 (V100)")
    elif 'A100' in gpu_name or 'A10' in gpu_name:
        BATCH_SIZE = 48
        print("Recommended batch size: 48 (A100)")
    else:
        BATCH_SIZE = 16
        print("Recommended batch size: 16")
else:
    print("No GPU detected!")
    BATCH_SIZE = 8

### Clone Code from GitHub

First time: Clones repository  
Updates: Pulls latest changes

In [ ]:
import os

# Check if already cloned
if os.path.exists(f"{PROJECT_DIR}/.git"):
    print("Repository already exists. Pulling latest changes...")
    %cd {PROJECT_DIR}
    !git pull
    print("Code updated from GitHub")
else:
    print("Cloning repository from GitHub...")
    !git clone {GITHUB_REPO} {PROJECT_DIR}
    print("Code cloned from GitHub")

# Verify backend folder exists
if os.path.exists(f"{PROJECT_DIR}/backend"):
    print(f"Backend code found at {PROJECT_DIR}/backend")
else:
    print("Backend folder not found!")
    print("Make sure you pushed the 'backend/' folder to GitHub")

### Download Dataset from Kaggle

Downloads from: https://www.kaggle.com/datasets/lakshaymiddha/crack-segmentation-dataset

In [ ]:
import os
import zipfile

dataset_dir = f"{PROJECT_DIR}/data/crack_segmentation_dataset"

# Check if dataset exists
if os.path.exists(dataset_dir) and os.path.exists(f"{dataset_dir}/train/images"):
    print("Dataset already exists in Google Drive!")
    print(f"  Location: {dataset_dir}")
    
    # Count files
    train_images = len(os.listdir(f"{dataset_dir}/train/images")) if os.path.exists(f"{dataset_dir}/train/images") else 0
    test_images = len(os.listdir(f"{dataset_dir}/test/images")) if os.path.exists(f"{dataset_dir}/test/images") else 0
    print(f"  Train images: {train_images}")
    print(f"  Test images: {test_images}")
else:
    print("Downloading dataset from Kaggle...")
    print("This takes 5-10 minutes but only happens ONCE.")
    print("Dataset will persist in Google Drive for all future sessions.\n")
    
    # Install Kaggle library
    print("Installing Kaggle library...")
    !pip install -q kaggle
    
    # Setup Kaggle credentials
    print("\nSetting up Kaggle API...")
    print("You need to upload your kaggle.json file.")
    print("Get it from: https://www.kaggle.com/settings -> API -> Create New API Token\n")
    
    from google.colab import files
    uploaded = files.upload()
    
    # Move kaggle.json to proper location
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    
    print("\nKaggle API configured successfully.")
    
    # Download dataset
    print("\nDownloading crack segmentation dataset from Kaggle...")
    !kaggle datasets download -d lakshaymiddha/crack-segmentation-dataset
    
    # Create directories
    os.makedirs(f"{PROJECT_DIR}/data", exist_ok=True)
    
    # Extract dataset
    print("\nExtracting dataset...")
    zip_file = "crack-segmentation-dataset.zip"
    
    with zipfile.ZipFile(zip_file, 'r') as zip_ref:
        zip_ref.extractall(f"{PROJECT_DIR}/data")
    
    # Remove zip file
    os.remove(zip_file)
    
    # Verify extraction
    extracted_files = os.listdir(f"{PROJECT_DIR}/data")
    print(f"\nExtracted contents: {extracted_files}")
    
    # Handle different possible extraction structures
    if 'crack_segmentation_dataset' in extracted_files:
        dataset_root = f"{PROJECT_DIR}/data/crack_segmentation_dataset"
    elif 'train' in extracted_files:
        # Dataset extracted directly to data/
        dataset_root = f"{PROJECT_DIR}/data"
        # Rename to expected structure
        os.rename(f"{PROJECT_DIR}/data", dataset_dir)
        os.makedirs(f"{PROJECT_DIR}/data", exist_ok=True)
        os.rename(dataset_dir, f"{PROJECT_DIR}/data/crack_segmentation_dataset")
        dataset_root = f"{PROJECT_DIR}/data/crack_segmentation_dataset"
    else:
        # Check subdirectories
        for item in extracted_files:
            item_path = f"{PROJECT_DIR}/data/{item}"
            if os.path.isdir(item_path) and os.path.exists(f"{item_path}/train"):
                dataset_root = item_path
                break
    
    # Count files
    train_images = len(os.listdir(f"{dataset_root}/train/images")) if os.path.exists(f"{dataset_root}/train/images") else 0
    test_images = len(os.listdir(f"{dataset_root}/test/images")) if os.path.exists(f"{dataset_root}/test/images") else 0
    
    print(f"\nDataset downloaded and saved to Google Drive!")
    print(f"  Location: {dataset_root}")
    print(f"  Train images: {train_images}")
    print(f"  Test images: {test_images}")
    print("  This dataset will persist across all Colab sessions.")

### Install Dependencies

In [ ]:
%cd {PROJECT_DIR}/backend

print("Installing dependencies...")
!pip install -q -r requirements.txt

print("Dependencies installed")

### Test Setup

In [ ]:
%cd {PROJECT_DIR}/backend

# Run test script
!python test_setup.py

### Start Training

In [ ]:
%cd {PROJECT_DIR}/backend

# Start fresh training
!python train.py \
    --config colab \
    --batch_size {BATCH_SIZE} \
    --num_epochs {NUM_EPOCHS} \
    --data_root {PROJECT_DIR}/data/crack_segmentation_dataset \
    --checkpoint_dir {PROJECT_DIR}/checkpoints \
    --log_dir {PROJECT_DIR}/logs \
    --sample_dir {PROJECT_DIR}/samples

### Resume Training After Disconnect

In [ ]:
import glob
import os

%cd {PROJECT_DIR}/backend

# Find latest checkpoint
checkpoint_pattern = f"{PROJECT_DIR}/checkpoints/checkpoint_epoch_*.pt"
checkpoints = sorted(glob.glob(checkpoint_pattern))

if checkpoints:
    latest_checkpoint = checkpoints[-1]
    epoch_num = os.path.basename(latest_checkpoint).split('_')[-1].replace('.pt', '')
    print(f"Found checkpoint: {os.path.basename(latest_checkpoint)}")
    print(f"Resuming from epoch {epoch_num}")
    print()
    
    # Resume training
    !python train.py \
        --resume {latest_checkpoint} \
        --config colab \
        --batch_size {BATCH_SIZE} \
        --num_epochs {NUM_EPOCHS} \
        --data_root {PROJECT_DIR}/data/crack_segmentation_dataset \
        --checkpoint_dir {PROJECT_DIR}/checkpoints \
        --log_dir {PROJECT_DIR}/logs \
        --sample_dir {PROJECT_DIR}/samples
else:
    print("No checkpoints found.")
    print("Use the 'Start Training' cell to start training from scratch.")

### Monitor Training - View Samples

In [ ]:
from IPython.display import Image, display
import glob

# Find latest samples
sample_pattern = f"{PROJECT_DIR}/samples/samples_epoch_*.png"
samples = sorted(glob.glob(sample_pattern))

if samples:
    latest = samples[-1]
    epoch = os.path.basename(latest).split('_')[-1].replace('.png', '')
    print(f"Latest samples (Epoch {epoch}):")
    display(Image(latest, width=800))
else:
    print("No samples generated yet.")
    print("Samples are generated every 5 epochs.")
    print("Wait for epoch 5, then re-run this cell.")

### Monitor Training - TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {PROJECT_DIR}/logs

### Check Training Progress

In [ ]:
import glob
import os

# Check checkpoints
checkpoints = sorted(glob.glob(f"{PROJECT_DIR}/checkpoints/checkpoint_epoch_*.pt"))
print(f"Training Progress\n" + "="*50)

if checkpoints:
    latest = checkpoints[-1]
    epoch = os.path.basename(latest).split('_')[-1].replace('.pt', '')
    print(f"Current epoch: {epoch}/{NUM_EPOCHS}")
    print(f"Progress: {int(epoch)/NUM_EPOCHS*100:.1f}%")
    print(f"Checkpoints saved: {len(checkpoints)}")
    print(f"\nLatest checkpoint: {os.path.basename(latest)}")
else:
    print("No checkpoints found yet.")
    print("Training may have just started.")

# Check if best model exists
if os.path.exists(f"{PROJECT_DIR}/checkpoints/best_model.pt"):
    print("\nBest model saved")

# Check samples
samples = glob.glob(f"{PROJECT_DIR}/samples/*.png")
if samples:
    print(f"\nSamples generated: {len(samples)}")

### Generate Images After Training - Inference

In [ ]:
%cd {PROJECT_DIR}/backend

# Generate from a test mask
CHECKPOINT = f"{PROJECT_DIR}/checkpoints/best_model.pt"
TEST_MASK = f"{PROJECT_DIR}/data/crack_segmentation_dataset/test/masks/00001.jpg"
OUTPUT_DIR = f"{PROJECT_DIR}/generated"

!python inference.py \
    --checkpoint {CHECKPOINT} \
    --mask_path {TEST_MASK} \
    --output_dir {OUTPUT_DIR} \
    --num_samples 4 \
    --num_steps 100

# Display result
from IPython.display import Image, display
import glob

generated = glob.glob(f"{OUTPUT_DIR}/*.png")
if generated:
    print("\nGenerated images:")
    display(Image(generated[-1], width=800))

### Download Results

In [ ]:
from google.colab import files
import shutil
import os

# What to download
print("Creating download archives...\n")

# 1. Best model
if os.path.exists(f"{PROJECT_DIR}/checkpoints/best_model.pt"):
    print("Downloading best model...")
    files.download(f"{PROJECT_DIR}/checkpoints/best_model.pt")
else:
    print("Best model not found")

# 2. All checkpoints (zipped)
if os.path.exists(f"{PROJECT_DIR}/checkpoints"):
    print("Creating checkpoints archive...")
    shutil.make_archive('/content/checkpoints', 'zip', f"{PROJECT_DIR}/checkpoints")
    files.download('/content/checkpoints.zip')

# 3. Generated samples (zipped)
if os.path.exists(f"{PROJECT_DIR}/samples"):
    print("Creating samples archive...")
    shutil.make_archive('/content/samples', 'zip', f"{PROJECT_DIR}/samples")
    files.download('/content/samples.zip')

print("\nDownloads complete!")

### Utilities - Update Code from GitHub

In [ ]:
%cd {PROJECT_DIR}
!git pull
print("Code updated from GitHub")

### Clear Old Checkpoints (Save Drive Space)

In [ ]:
# Keep only best_model.pt and final_model.pt
# Delete intermediate checkpoints

import glob
import os

intermediate = glob.glob(f"{PROJECT_DIR}/checkpoints/checkpoint_epoch_*.pt")
print(f"Found {len(intermediate)} intermediate checkpoints")
print("\nDelete them? (keeps best_model.pt and final_model.pt)")
print("Uncomment the line below to delete:")

# Uncomment to delete:
# for cp in intermediate:
#     os.remove(cp)
#     print(f"Deleted: {os.path.basename(cp)}")

### Check GPU Memory Usage

In [ ]:
!nvidia-smi